In [3]:
import json
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# -----------------------------------------
# PATHS
# -----------------------------------------

INPUT_DIR = Path("data/enriched")
OUTPUT_DIR = Path("data/chunks")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------
# FAST CHUNKER
# -----------------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

# -----------------------------------------
# PROCESS ALL FILES
# -----------------------------------------

json_files = sorted(INPUT_DIR.glob("*.json"))

if not json_files:
    raise FileNotFoundError(
        f"No JSON files found in {INPUT_DIR}"
    )

total_documents = 0
total_pages = 0
total_chunks = 0

for input_file in json_files:

    print(f"\nProcessing: {input_file.name}")

    try:
        with open(input_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        doc_id = data["doc_id"]
        metadata = data["metadata"]
        pages = data["parsed_content"]["pages"]

        chunks = []
        chunk_counter = 1
        pages_processed = 0

        # -----------------------------------------
        # PROCESS EACH PDF PAGE
        # -----------------------------------------

        for page_data in pages:

            text = page_data.get("text", "").strip()

            if not text:
                continue

            page_number = page_data["page"]
            pages_processed += 1

            page_document = Document(
                page_content=text,
                metadata={
                    "doc_id": doc_id,
                    "company": metadata["company"],
                    "sector": metadata["sector"],
                    "desk": metadata["desk"],
                    "doc_type": metadata["doc_type"],
                    "period": metadata["period"],
                    "page": page_number,
                    "source": data["parsed_content"]["source_file"]
                }
            )

            page_chunks = splitter.split_documents(
                [page_document]
            )

            # -----------------------------------------
            # ADD METADATA
            # -----------------------------------------

            for chunk in page_chunks:

                original_text = chunk.page_content

                chunk_id = (
                    f"{doc_id}_p{page_number}"
                    f"_c{chunk_counter:04d}"
                )

                contextual_header = (
                    f"Company: {metadata['company']}\n"
                    f"Filing: {metadata['doc_type']}\n"
                    f"Reporting period: {metadata['period']}\n"
                    f"Document ID: {doc_id}\n"
                    f"PDF page: {page_number}\n"
                )

                chunks.append({
                    "chunk_id": chunk_id,
                    "doc_id": doc_id,
                    "company": metadata["company"],
                    "sector": metadata["sector"],
                    "desk": metadata["desk"],
                    "doc_type": metadata["doc_type"],
                    "period": metadata["period"],
                    "page": page_number,
                    "source": data["parsed_content"]["source_file"],
                    "content_type": "text",

                    # Clean evidence for answer generation
                    "original_text": original_text,

                    # Contextualized text for embedding
                    "page_content": (
                        contextual_header
                        + "\nContent:\n"
                        + original_text
                    )
                })

                chunk_counter += 1

        # -----------------------------------------
        # SAVE
        # -----------------------------------------

        output_data = {
            "doc_id": doc_id,
            "metadata": metadata,
            "total_pages_processed": pages_processed,
            "total_chunks": len(chunks),
            "chunks": chunks
        }

        output_path = (
            OUTPUT_DIR / f"{doc_id}_chunks.json"
        )

        with open(
            output_path,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                output_data,
                f,
                ensure_ascii=False,
                indent=2
            )

        total_documents += 1
        total_pages += pages_processed
        total_chunks += len(chunks)

        print(f"Pages:  {pages_processed}")
        print(f"Chunks: {len(chunks)}")
        print(f"Saved:  {output_path}")

    except Exception as e:

        print(
            f"ERROR processing {input_file.name}: {e}"
        )

# -----------------------------------------
# SUMMARY
# -----------------------------------------

print("\n" + "=" * 60)
print("CHUNKING COMPLETED")
print("=" * 60)

print("Documents:", total_documents)
print("Pages:", total_pages)
print("Chunks:", total_chunks)
print("Output:", OUTPUT_DIR)


Processing: 3M_2015_10K.json
Pages:  158
Chunks: 794
Saved:  data\chunks\3M_2015_10K_chunks.json

Processing: 3M_2016_10K.json
Pages:  233
Chunks: 943
Saved:  data\chunks\3M_2016_10K_chunks.json

Processing: 3M_2017_10K.json
Pages:  176
Chunks: 797
Saved:  data\chunks\3M_2017_10K_chunks.json

Processing: 3M_2018_10K.json
Pages:  160
Chunks: 800
Saved:  data\chunks\3M_2018_10K_chunks.json

Processing: 3M_2019_10K.json
Pages:  186
Chunks: 914
Saved:  data\chunks\3M_2019_10K_chunks.json

Processing: 3M_2020_10K.json
Pages:  165
Chunks: 815
Saved:  data\chunks\3M_2020_10K_chunks.json

Processing: 3M_2021_10K.json
Pages:  161
Chunks: 791
Saved:  data\chunks\3M_2021_10K_chunks.json

Processing: 3M_2022_10K.json
Pages:  252
Chunks: 1232
Saved:  data\chunks\3M_2022_10K_chunks.json

Processing: 3M_2023Q2_10Q.json
Pages:  92
Chunks: 478
Saved:  data\chunks\3M_2023Q2_10Q_chunks.json

Processing: ADOBE_2015_10K.json
Pages:  116
Chunks: 621
Saved:  data\chunks\ADOBE_2015_10K_chunks.json

Processin

In [4]:
import hashlib

from qdrant_client import QdrantClient, models
from langchain_qdrant import QdrantVectorStore

import os
from dotenv import load_dotenv

from qdrant_client import QdrantClient, models
from langchain_qdrant import QdrantVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

COLLECTION_NAME = "sec_filings"
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True
    }
)
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

if not client.collection_exists(COLLECTION_NAME):

    vector_size = len(
        embeddings.embed_query("Test embedding")
    )

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE
        )
    )

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings
)

print("Connected to remote Qdrant successfully!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2027.17it/s]


Connected to remote Qdrant successfully!


In [6]:
import os
import json
import uuid
import hashlib
from pathlib import Path

from dotenv import load_dotenv
from tqdm import tqdm

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from qdrant_client import QdrantClient, models
from langchain_qdrant import QdrantVectorStore


# =========================================================
# CONFIG
# =========================================================

load_dotenv()

CHUNKS_DIR = Path("data/chunks")
COLLECTION_NAME = "sec_filings"

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

BATCH_SIZE = 64


# =========================================================
# VALIDATE ENV
# =========================================================

if not QDRANT_URL:
    raise ValueError("QDRANT_URL is not set")

if not QDRANT_API_KEY:
    raise ValueError("QDRANT_API_KEY is not set")


# =========================================================
# EMBEDDINGS
# =========================================================

print("Loading BGE embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding model loaded.")


# =========================================================
# QDRANT CLIENT
# =========================================================

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=120
)


# =========================================================
# CREATE COLLECTION
# =========================================================

if not client.collection_exists(COLLECTION_NAME):

    print(f"Creating collection: {COLLECTION_NAME}")

    vector_size = len(
        embeddings.embed_query("test embedding")
    )

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE
        )
    )

else:

    print(f"Collection already exists: {COLLECTION_NAME}")


# =========================================================
# PAYLOAD INDEXES
# =========================================================

for field_name in [
    "metadata.company",
    "metadata.doc_id",
    "metadata.desk",
    "metadata.doc_type",
]:

    try:

        client.create_payload_index(
            collection_name=COLLECTION_NAME,
            field_name=field_name,
            field_schema=models.PayloadSchemaType.KEYWORD
        )

    except Exception:
        pass


# =========================================================
# VECTOR STORE
# =========================================================

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings
)


# =========================================================
# LOAD CHUNK FILES
# =========================================================

chunk_files = sorted(
    CHUNKS_DIR.glob("*_chunks.json")
)

if not chunk_files:
    raise FileNotFoundError(
        f"No chunk files found in {CHUNKS_DIR}"
    )

print(f"\nFound {len(chunk_files)} chunk files.")


# =========================================================
# INGEST EACH DOCUMENT
# =========================================================

total_documents = 0
total_chunks = 0

for chunk_file in chunk_files:

    print("\n" + "=" * 60)
    print(f"INGESTING: {chunk_file.name}")
    print("=" * 60)

    with open(
        chunk_file,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    doc_id = data["doc_id"]
    chunks = data["chunks"]

    if not chunks:
        print("No chunks found. Skipping.")
        continue

    # -----------------------------------------------------
    # DELETE OLD VERSION OF THIS DOCUMENT
    # -----------------------------------------------------

    print(f"Removing existing vectors for {doc_id}...")

    client.delete(
        collection_name=COLLECTION_NAME,
        points_selector=models.FilterSelector(
            filter=models.Filter(
                must=[
                    models.FieldCondition(
                        key="metadata.doc_id",
                        match=models.MatchValue(
                            value=doc_id
                        )
                    )
                ]
            )
        ),
        wait=True
    )

    # -----------------------------------------------------
    # BUILD DOCUMENT OBJECTS
    # -----------------------------------------------------

    documents = []
    point_ids = []

    for chunk in chunks:

        documents.append(
            Document(
                page_content=chunk["page_content"],
                metadata={
                    "doc_id": chunk["doc_id"],
                    "company": chunk["company"],
                    "sector": chunk["sector"],
                    "desk": chunk["desk"],
                    "doc_type": chunk["doc_type"],
                    "period": chunk["period"],
                    "page": chunk["page"],
                    "source": chunk["source"],
                    "chunk_id": chunk["chunk_id"],
                    "content_type": chunk["content_type"],
                    "original_text": chunk["original_text"],
                }
            )
        )

        # -------------------------------------------------
        # DETERMINISTIC ID
        # -------------------------------------------------

        content_hash = hashlib.sha256(
            chunk["original_text"].encode("utf-8")
        ).hexdigest()

        point_id = str(
            uuid.uuid5(
                uuid.NAMESPACE_URL,
                f"{doc_id}:"
                f"{chunk['page']}:"
                f"{content_hash}"
            )
        )

        point_ids.append(point_id)

    # -----------------------------------------------------
    # BATCH INSERT
    # -----------------------------------------------------

    for start in tqdm(
        range(
            0,
            len(documents),
            BATCH_SIZE
        ),
        desc=f"Embedding {doc_id}"
    ):

        end = min(
            start + BATCH_SIZE,
            len(documents)
        )

        batch_documents = documents[start:end]
        batch_ids = point_ids[start:end]

        vector_store.add_documents(
            documents=batch_documents,
            ids=batch_ids,
            batch_size=BATCH_SIZE
        )

    total_documents += 1
    total_chunks += len(documents)

    print(f"Inserted chunks: {len(documents)}")


# =========================================================
# FINAL SUMMARY
# =========================================================

print("\n" + "=" * 60)
print("INGESTION COMPLETED")
print("=" * 60)

print("Documents:", total_documents)
print("Chunks:", total_chunks)
print("Collection:", COLLECTION_NAME)

Loading BGE embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1718.53it/s]


Embedding model loaded.
Collection already exists: sec_filings

Found 21 chunk files.

INGESTING: PFIZER_2018_10K_chunks.json
Removing existing vectors for PFIZER_2018_10K...


Embedding PFIZER_2018_10K: 100%|██████████| 38/38 [13:13<00:00, 20.88s/it]


Inserted chunks: 2380

INGESTING: PFIZER_2019_10K_chunks.json
Removing existing vectors for PFIZER_2019_10K...


Embedding PFIZER_2019_10K: 100%|██████████| 22/22 [07:01<00:00, 19.16s/it]


... (35 outputs left out)

Embedding VERIZON_2021_10K: 100%|██████████| 10/10 [05:04<00:00, 30.47s/it]


Inserted chunks: 622

INGESTING: VERIZON_2022_10K_chunks.json
Removing existing vectors for VERIZON_2022_10K...


Embedding VERIZON_2022_10K: 100%|██████████| 10/10 [04:43<00:00, 28.35s/it]

Inserted chunks: 617

INGESTION COMPLETED
Documents: 21
Chunks: 13657
Collection: sec_filings


In [ ]:

import json
import os
import re
from collections import defaultdict

import cohere
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

load_dotenv()

# --------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------
COLLECTION_NAME = "sec_filings"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"          # must match the indexing model
RERANK_MODEL = os.getenv("COHERE_RERANK_MODEL", "rerank-v5.0-pro")
GENERATION_MODEL = os.getenv("COHERE_GENERATION_MODEL", "command-a-03-2025")

ROLE_ACCESS = {
    "consumer": ["PepsiCo", "Best Buy", "MGM Resorts", "Ulta Beauty"],
    "health_industrials": ["Johnson & Johnson", "Pfizer", "3M", "Boeing"],
    "tech_telecom": ["AMD", "Adobe", "Corning", "Verizon"],
    "lead": [
        "PepsiCo", "Best Buy", "MGM Resorts", "Ulta Beauty",
        "Johnson & Johnson", "Pfizer", "3M", "Boeing",
        "AMD", "Adobe", "Corning", "Verizon",
    ],
}

# --------------------------------------------------------------------------
# Clients
# --------------------------------------------------------------------------
co = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))

client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 892.53it/s]


In [ ]:

def ensure_indexes():
    """Payload indexes used by filters (safe to call repeatedly)."""
    wanted = {
        "metadata.company": models.PayloadSchemaType.KEYWORD,
        "metadata.doc_id": models.PayloadSchemaType.KEYWORD,
        "metadata.page": models.PayloadSchemaType.INTEGER,
    }
    for field, schema in wanted.items():
        try:
            client.create_payload_index(COLLECTION_NAME, field_name=field, field_schema=schema)
        except Exception:
            pass  # already exists


ensure_indexes()


# --------------------------------------------------------------------------
# 1. Query rewriting
# --------------------------------------------------------------------------
FALLBACK_KEYWORDS = (
    "litigation, legal proceedings, legal claims, regulatory proceedings, lawsuits, "
    "investigations, inquiries, potential liabilities, material adverse effect"
)

REWRITE_SYSTEM = """You rewrite questions for searching SEC filings (10-K, 10-Q, 8-K).
Return ONLY a JSON object: {"queries": ["...", "...", "..."]}

Rules:
- Produce 3 to 4 short search queries that together cover the question.
- Use the formal wording and section names found in SEC filings
  (e.g. "Item 3. Legal Proceedings", "Commitments and Contingencies",
  "Item 7. Management's Discussion and Analysis", "Item 1A. Risk Factors").
- Keep company names, fiscal years and financial terms from the question.
- If the question names several years, include at least one query per year.
- Do not answer the question."""


def rewrite_query(question, n_max=4):
    """Return [original question, + LLM rewrites]. Falls back to keyword expansion."""
    queries = [question]
    try:
        resp = co.chat(
            model=GENERATION_MODEL,
            messages=[
                {"role": "system", "content": REWRITE_SYSTEM},
                {"role": "user", "content": question},
            ],
            response_format={"type": "json_object"},
            temperature=0.0,
        )
        data = json.loads(resp.message.content[0].text)
        queries += [q for q in data.get("queries", []) if isinstance(q, str)][:n_max]
    except Exception:
        queries.append(f"{question}\nTopics: {FALLBACK_KEYWORDS}")

    seen, unique = set(), []
    for q in queries:  # de-duplicate, keep order
        key = q.strip().lower()
        if key and key not in seen:
            seen.add(key)
            unique.append(q.strip())
    return unique



In [9]:
def retrieve(queries, role, k_per_query=30):
    if role not in ROLE_ACCESS:
        raise ValueError(f"Invalid role: {role}")

    access_filter = models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.company",
                match=models.MatchAny(any=ROLE_ACCESS[role]),
            )
        ]
    )

    best = {}  # chunk_id -> candidate (keep best dense score)
    for q in queries:
        for doc, score in vector_store.similarity_search_with_score(
            query=q, k=k_per_query, filter=access_filter
        ):
            m = doc.metadata
            cid = m["chunk_id"]
            if cid not in best or score > best[cid]["dense_score"]:
                best[cid] = {
                    "doc_id": m["doc_id"],
                    "company": m.get("company", ""),
                    "doc_type": m.get("doc_type", ""),
                    "period": m.get("period", ""),
                    "page": m["page"],
                    "chunk_id": cid,
                    "dense_score": float(score),
                    "text": m.get("original_text", doc.page_content),
                }
    return sorted(best.values(), key=lambda c: c["dense_score"], reverse=True)


In [ ]:
def rerank(question, candidates, top_n=10):
    if not candidates:
        return []

    # Prefix with light context so the reranker knows which filing/page it is reading
    docs = [
        f"{c['company']} {c['doc_type']} {c['period']} | PDF page {c['page']}\n{c['text']}"
        for c in candidates
    ]
    resp = co.rerank(
        model=RERANK_MODEL,
        query=question,                      # original question, not the expanded one
        documents=docs,
        top_n=min(top_n, len(docs)),
    )
    return [
        {**candidates[r.index], "rerank_rank": rank, "rerank_score": float(r.relevance_score)}
        for rank, r in enumerate(resp.results, start=1)
    ]
def build_context(evidence):
    blocks = []
    for i, e in enumerate(evidence, start=1):
        blocks.append(
            f"[Evidence {i}] Document ID: {e['doc_id']} | PDF page: {e['page']} | "
            f"Chunk: {e['chunk_id']}\n{e['text'].strip()}"
        )
    return "\n\n---\n\n".join(blocks)


# --------------------------------------------------------------------------
# 4. Generation
# --------------------------------------------------------------------------
SYSTEM_PROMPT = """You are an expert financial research analyst for SEC filings (10-K, 10-Q, 8-K, earnings reports).
Answer ONLY from the evidence blocks supplied in the user message.

EVIDENCE RULES
- Never use outside knowledge, assumptions, or memory. Never invent facts, figures, events, or citations.
- Every factual claim needs a citation in the form [DOCUMENT_ID, PDF page X], using the exact
  document ID and page shown in the evidence block it came from.
- If evidence blocks conflict, state the conflict instead of choosing silently.

INTERPRETATION RULES
- Separate these ideas: (a) a matter exists, (b) it is individually material, (c) management's
  expected aggregate financial impact. "No material adverse effect expected" is a statement about (c).
  It does NOT mean no proceedings exist, and it does NOT prove that no single case is material.
  Report management's wording and scope exactly; if materiality of individual matters is not
  established, say: "The filing does not establish whether any individual proceeding is
  independently material."
- Distinguish risk-factor language ("could", "may") from disclosed actual events, and distinguish
  ongoing, historical, and completed matters.
- A 10-K for fiscal year N normally also contains comparative data for year N-1. Treat each
  requested period separately; only attribute a fact to a period when the evidence supports it.
  If a requested period is not supported, say it cannot be verified from the evidence.
- Absence of retrieved evidence is NOT evidence that something did not happen.

NUMBERS
- Preserve original units, currencies, and periods. Show inputs and formula for any calculation.
  Never combine figures from different periods without explaining the comparison.

OUTPUT FORMAT
If the evidence supports an answer:
STATUS: ANSWERED
ANSWER: <direct answer, then short explanation, per period when several are asked>
EVIDENCE:
- <finding> [DOCUMENT_ID, PDF page X]
LIMITATIONS: <gaps or caveats, or "None identified">

If it does not:
STATUS: REFUSED
ANSWER: The available filing evidence is insufficient to answer this question.
REASON: <what is missing>"""


def generate(question, evidence):
    if not evidence:
        return "STATUS: REFUSED\nANSWER: No relevant filing evidence was retrieved.\nREASON: Retrieval returned nothing."

    user_msg = (
        f"QUESTION:\n{question}\n\n"
        f"RETRIEVED SEC FILING EVIDENCE:\n{build_context(evidence)}\n\n"
        "Answer the exact question asked, period by period, using only this evidence."
    )
    resp = co.chat(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.0,
    )
    return resp.message.content[0].text


# --------------------------------------------------------------------------
# End-to-end
# --------------------------------------------------------------------------
def ask_filings(
    question,
    role,
    k_per_query=30,
    top_n=8,
    verbose=True,
):
    queries = rewrite_query(question)
    candidates = retrieve(queries, role, k_per_query=k_per_query)
    evidence = rerank(question, candidates, top_n=top_n)   # top reranked chunks go straight to the LLM
    answer = generate(question, evidence)

    if verbose:
        print("Queries:")
        for q in queries:
            print("  -", q.replace("\n", " ")[:120])
        print(f"\nCandidates: {len(candidates)} | Evidence chunks: {len(evidence)}")
        for e in evidence:
            print(f"  #{e['rerank_rank']:<3} p{e['page']:<4} {e['chunk_id']:<32} rerank={e['rerank_score']:.4f}")

    return {
        "answer": answer,
        "queries": queries,
        "citations": sorted({(e["doc_id"], e["page"]) for e in evidence}),
        "evidence": evidence,
    }


if __name__ == "__main__":
    question = (
        ""
    )
    result = ask_filings(question, role="consumer")
    print("\n" + "=" * 70)
    print(result["answer"])


Queries:
  - Basing your judgments off of the balance sheet, what is the year end FY2018 amount of accounts payable for MGM Resorts? 
  - FY2018 balance sheet accounts payable
  - MGM Resorts FY2018 accounts payable
  - 2018 MGM Resorts balance sheet liabilities

Candidates: 117 | Evidence chunks: 8
  #1   p64   MGMRESORTS_2018_10K_p64_c0354    rerank=0.2137
  #2   p11   MGMRESORTS_2023Q2_10Q_p11_c0043  rerank=0.2059
  #3   p79   MGMRESORTS_2022_10K_p79_c0417    rerank=0.1859
  #4   p176  MGMRESORTS_2018_10K_p176_c0844   rerank=0.1856
  #5   p48   MGMRESORTS_2018_10K_p48_c0277    rerank=0.1792
  #6   p56   MGMRESORTS_2018_10K_p56_c0315    rerank=0.1606
  #7   p55   MGMRESORTS_2019_10K_p55_c0315    rerank=0.1420
  #8   p82   MGMRESORTS_2018_10K_p82_c0443    rerank=0.1228

STATUS: REFUSED
ANSWER: The available filing evidence is insufficient to answer this question.
REASON: The provided evidence does not contain the specific figure for accounts payable for MGM Resorts at the end of FY201

In [16]:
"""
Cohere RAG over SEC filings stored in Qdrant.

Pipeline
    1. Query rewriting   (Cohere chat -> several SEC-style search queries)
    2. Dense retrieval   (BGE embeddings + Qdrant, role-based filter, multi-query, dedupe)
    3. Reranking         (Cohere Rerank on the ORIGINAL question)
    4. Generation        (Cohere chat, strict evidence-only prompt, temperature 0)

.env needs: QDRANT_URL, QDRANT_API_KEY, COHERE_API_KEY
Optional  : COHERE_RERANK_MODEL, COHERE_GENERATION_MODEL
"""

import json
import os
import re
from collections import defaultdict

import cohere
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

load_dotenv()

# --------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------
COLLECTION_NAME = "sec_filings"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"          # must match the indexing model
RERANK_MODEL = os.getenv("COHERE_RERANK_MODEL", "rerank-v5.0-pro")
GENERATION_MODEL = os.getenv("COHERE_GENERATION_MODEL", "command-a-03-2025")

ROLE_ACCESS = {
    "consumer": ["PepsiCo", "Best Buy", "MGM Resorts", "Ulta Beauty"],
    "health_industrials": ["Johnson & Johnson", "Pfizer", "3M", "Boeing"],
    "tech_telecom": ["AMD", "Adobe", "Corning", "Verizon"],
    "lead": [
        "PepsiCo", "Best Buy", "MGM Resorts", "Ulta Beauty",
        "Johnson & Johnson", "Pfizer", "3M", "Boeing",
        "AMD", "Adobe", "Corning", "Verizon",
    ],
}

# --------------------------------------------------------------------------
# Clients
# --------------------------------------------------------------------------
co = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))

client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)


def ensure_indexes():
    """Payload indexes used by filters (safe to call repeatedly)."""
    wanted = {
        "metadata.company": models.PayloadSchemaType.KEYWORD,
        "metadata.doc_id": models.PayloadSchemaType.KEYWORD,
        "metadata.page": models.PayloadSchemaType.INTEGER,
        # full-text index so financial-statement pages can be found by their title
        "page_content": models.TextIndexParams(
            type=models.TextIndexType.TEXT,
            tokenizer=models.TokenizerType.WORD,
            min_token_len=2,
            max_token_len=20,
            lowercase=True,
        ),
    }
    for field, schema in wanted.items():
        try:
            client.create_payload_index(COLLECTION_NAME, field_name=field, field_schema=schema)
        except Exception:
            pass  # already exists


ensure_indexes()


# --------------------------------------------------------------------------
# 1. Query rewriting
# --------------------------------------------------------------------------
FALLBACK_KEYWORDS = (
    "litigation, legal proceedings, legal claims, regulatory proceedings, lawsuits, "
    "investigations, inquiries, potential liabilities, material adverse effect"
)

REWRITE_SYSTEM = """You rewrite questions for searching SEC filings (10-K, 10-Q, 8-K).
Return ONLY a JSON object: {"queries": ["...", "...", "..."]}

Rules:
- Produce 3 to 4 short search queries that together cover the question.
- Use the formal wording and section names found in SEC filings
  (e.g. "Item 3. Legal Proceedings", "Commitments and Contingencies",
  "Item 7. Management's Discussion and Analysis", "Item 1A. Risk Factors").
- Keep company names, fiscal years and financial terms from the question.
- If the question refers to a balance sheet, income statement or cash flow statement, write one query
  with the statement's exact title, e.g. "MGM Resorts International consolidated balance sheets
  December 31, 2018 accounts payable total current liabilities".
- If the question names several years, include at least one query per year.
- Do not answer the question."""


def rewrite_query(question, n_max=4):
    """Return [original question, + LLM rewrites]. Falls back to keyword expansion."""
    queries = [question]
    try:
        resp = co.chat(
            model=GENERATION_MODEL,
            messages=[
                {"role": "system", "content": REWRITE_SYSTEM},
                {"role": "user", "content": question},
            ],
            response_format={"type": "json_object"},
            temperature=0.0,
        )
        data = json.loads(resp.message.content[0].text)
        queries += [q for q in data.get("queries", []) if isinstance(q, str)][:n_max]
    except Exception:
        queries.append(f"{question}\nTopics: {FALLBACK_KEYWORDS}")

    seen, unique = set(), []
    for q in queries:  # de-duplicate, keep order
        key = q.strip().lower()
        if key and key not in seen:
            seen.add(key)
            unique.append(q.strip())
    return unique


# --------------------------------------------------------------------------
# 2. Question filters + retrieval
# --------------------------------------------------------------------------
COMPANY_ALIASES = {
    "PepsiCo": ["pepsico", "pepsi"],
    "Best Buy": ["best buy"],
    "MGM Resorts": ["mgm resorts", "mgm"],
    "Ulta Beauty": ["ulta beauty", "ulta"],
    "Johnson & Johnson": ["johnson & johnson", "johnson and johnson", "jnj", "j&j"],
    "Pfizer": ["pfizer"],
    "3M": ["3m"],
    "Boeing": ["boeing"],
    "AMD": ["amd"],
    "Adobe": ["adobe"],
    "Corning": ["corning"],
    "Verizon": ["verizon"],
}

STATEMENT_TRIGGERS = {
    "balance_sheet": ["balance sheet"],
    "income_statement": ["income statement", "p&l", "statement of income", "statements of income",
                         "statement of operations", "statement of earnings", "profit and loss"],
    "cash_flow": ["cash flow", "cashflow", "statement of cash flows"],
}
STATEMENT_PHRASES = {   # words that must appear on the statement page
    "balance_sheet": ["balance sheets", "balance sheet"],
    "income_statement": ["statements of operations", "statement of operations", "statements of income",
                         "statement of income", "statements of earnings", "statement of earnings"],
    "cash_flow": ["statements of cash flows", "statement of cash flows"],
}
STATEMENT_TITLE_RE = re.compile(
    r"consolidated\s+(balance\s+sheets?|statements?\s+of\s+(operations|income|earnings|cash\s+flows))",
    re.I,
)


def extract_filters(question, role):
    """Rule-based: companies, fiscal years and financial statements mentioned in the question."""
    q = question.lower()

    companies = [
        c for c in ROLE_ACCESS[role]
        if any(re.search(rf"(?<![a-z0-9]){re.escape(a)}(?![a-z0-9])", q)
               for a in COMPANY_ALIASES.get(c, [c.lower()]))
    ]

    years = {int(y) for y in re.findall(r"(?<!\d)((?:19|20)\d{2})(?!\d)", q)}
    years |= {2000 + int(y) for y in re.findall(r"\bfy\s*'?(\d{2})\b", q)}
    years = {y for y in years if 1990 <= y <= 2035}

    statements = [st for st, keys in STATEMENT_TRIGGERS.items() if any(k in q for k in keys)]

    # does the question ask about a quarter / half-year? If not, interim filings are noise
    interim = bool(re.search(r"\b(q[1-4]|h[12]|quarter\w*|half|semi-?annual|interim)\b", q))

    return {"companies": companies, "years": sorted(years), "statements": statements, "interim": interim}


def year_window(years):
    """FY N also appears in the FY N+1 10-K (comparatives); forecasts for N+1 sit in the FY N 10-K."""
    return {y + d for y in years for d in (-1, 0, 1)}


INTERIM_RE = re.compile(r"_\d{4}Q[123](?:_|$)")           # Q1-Q3 filings (Q4 = year end, kept)
DOC_RE = re.compile(r"_(\d{4})(Q[1-4])?_(10K|10Q|8K|EARNINGS)$")


def describe_doc(doc_id):
    """ULTABEAUTY_2023Q4_EARNINGS -> 'Q4 earnings release (labelled 2023)'"""
    m = DOC_RE.search(doc_id)
    if not m:
        return doc_id
    year, q, kind = m.groups()
    names = {"10K": "annual report (10-K)", "10Q": "quarterly report (10-Q)",
             "8K": "8-K", "EARNINGS": "earnings release"}
    return f"{(q + ' ') if q else ''}{names[kind]} (labelled {year})"


def doc_in_window(doc_id, window):
    return any(re.search(rf"_{y}(?!\d)", doc_id) for y in window)


def _candidate(doc_id, page, chunk_id, text, meta, score):
    return {
        "doc_id": doc_id,
        "company": meta.get("company", ""),
        "doc_type": meta.get("doc_type", ""),
        "period": meta.get("period", ""),
        "page": page,
        "chunk_id": chunk_id,
        "dense_score": float(score),
        "text": text,
    }


def fetch_page_chunks(doc_id, page):
    """Every chunk of one PDF page (a long table can be split in several chunks)."""
    flt = models.Filter(must=[
        models.FieldCondition(key="metadata.doc_id", match=models.MatchValue(value=doc_id)),
        models.FieldCondition(key="metadata.page", match=models.MatchValue(value=page)),
    ])
    points, _ = client.scroll(COLLECTION_NAME, scroll_filter=flt, limit=64,
                              with_payload=True, with_vectors=False)
    out = []
    for p in points:
        m = p.payload["metadata"]
        out.append(_candidate(m["doc_id"], m["page"], m["chunk_id"],
                              m.get("original_text", p.payload.get("page_content", "")), m, 0.0))
    return out


def lookup_statement_pages(companies, statements, years, max_pages=12):
    """
    Dense search often misses table-heavy statement pages. Find them directly:
    full-text match on the statement title, restricted to the company and the
    filing years around the question's fiscal year, then return ALL chunks of those pages.
    """
    if not (companies and statements and years):
        return []
    window = year_window(years)

    pairs = set()
    for st in statements:
        flt = models.Filter(
            must=[models.FieldCondition(key="metadata.company", match=models.MatchAny(any=companies))],
            should=[models.FieldCondition(key="page_content", match=models.MatchText(text=p))
                    for p in STATEMENT_PHRASES[st]],
        )
        offset = None
        while True:
            points, offset = client.scroll(COLLECTION_NAME, scroll_filter=flt, limit=256,
                                           offset=offset, with_payload=True, with_vectors=False)
            for p in points:
                m = p.payload["metadata"]
                text = m.get("original_text", "")
                # keep pages that START with the statement title (the real statement, not a mention)
                if doc_in_window(m["doc_id"], window) and STATEMENT_TITLE_RE.search(text[:400]):
                    pairs.add((m["doc_id"], m["page"]))
            if offset is None:
                break

    # prefer annual filings, then document order
    ranked = sorted(pairs, key=lambda x: (not x[0].endswith("10K"), x[0], x[1]))[:max_pages]
    out = []
    for doc_id, page in ranked:
        out.extend(fetch_page_chunks(doc_id, page))
    return out


def retrieve(queries, role, filters=None, k_per_query=30):
    if role not in ROLE_ACCESS:
        raise ValueError(f"Invalid role: {role}")
    filters = filters or {"companies": [], "years": [], "statements": []}

    def company_filter(companies):
        return models.Filter(must=[models.FieldCondition(
            key="metadata.company", match=models.MatchAny(any=companies))])

    best = {}  # chunk_id -> candidate (keep best dense score)

    def add(c):
        cid = c["chunk_id"]
        if cid not in best or c["dense_score"] > best[cid]["dense_score"]:
            best[cid] = c

    # (a) role-wide search, plus (b) a search narrowed to the company named in the question
    searches = [ROLE_ACCESS[role]]
    if filters["companies"]:
        searches.append(filters["companies"])
    for companies in searches:
        for q in queries:
            for doc, score in vector_store.similarity_search_with_score(
                query=q, k=k_per_query, filter=company_filter(companies)
            ):
                m = doc.metadata
                add(_candidate(m["doc_id"], m["page"], m["chunk_id"],
                               m.get("original_text", doc.page_content), m, score))

    # (c) direct lookup of the financial statement pages the question refers to
    try:
        for c in lookup_statement_pages(filters["companies"], filters["statements"], filters["years"]):
            add(c)
    except Exception as e:                       # never let the lookup break normal retrieval
        print("Statement lookup skipped:", repr(e))

    candidates = sorted(best.values(), key=lambda c: c["dense_score"], reverse=True)

    # (d) soft year filter: keep filings within +/-1 year of the fiscal year asked about
    if filters["years"]:
        window = year_window(filters["years"])
        in_window = [c for c in candidates if doc_in_window(c["doc_id"], window)]
        if len(in_window) >= 5:         # fall back to everything if the filter is too aggressive
            candidates = in_window

    # (e) year-end questions: drop Q1-Q3 filings unless the question asks about a quarter
    if not filters.get("interim"):
        annual = [c for c in candidates if not INTERIM_RE.search(c["doc_id"])]
        if len(annual) >= 5:
            candidates = annual

    return candidates


# --------------------------------------------------------------------------
# 3. Cohere rerank
# --------------------------------------------------------------------------
def rerank(question, candidates, top_n=10):
    if not candidates:
        return []

    # Prefix with light context so the reranker knows which filing/page it is reading
    docs = [
        f"{c['company']} {c['doc_type']} {c['period']} | PDF page {c['page']}\n{c['text']}"
        for c in candidates
    ]
    resp = co.rerank(
        model=RERANK_MODEL,
        query=question,                      # original question, not the expanded one
        documents=docs,
        top_n=min(top_n, len(docs)),
    )
    return [
        {**candidates[r.index], "rerank_rank": rank, "rerank_score": float(r.relevance_score)}
        for rank, r in enumerate(resp.results, start=1)
    ]


def build_context(evidence):
    blocks = []
    for i, e in enumerate(evidence, start=1):
        blocks.append(
            f"[Evidence {i}] Document ID: {e['doc_id']} | Filing: {describe_doc(e['doc_id'])} | "
            f"PDF page: {e['page']} | Chunk: {e['chunk_id']}\n{e['text'].strip()}"
        )
    return "\n\n---\n\n".join(blocks)


# --------------------------------------------------------------------------
# 4. Generation
# --------------------------------------------------------------------------
SYSTEM_PROMPT = """You are an expert financial research analyst for SEC filings (10-K, 10-Q, 8-K, earnings reports).
Answer ONLY from the evidence blocks supplied in the user message.

EVIDENCE RULES
- Never use outside knowledge, assumptions, or memory. Never invent facts, figures, events, or citations.
- Every factual claim needs a citation in the form [DOCUMENT_ID, PDF page X], using the exact
  document ID and page shown in the evidence block it came from.
- If evidence blocks conflict, state the conflict instead of choosing silently.

INTERPRETATION RULES
- Separate these ideas: (a) a matter exists, (b) it is individually material, (c) management's
  expected aggregate financial impact. "No material adverse effect expected" is a statement about (c).
  It does NOT mean no proceedings exist, and it does NOT prove that no single case is material.
  Report management's wording and scope exactly; if materiality of individual matters is not
  established, say: "The filing does not establish whether any individual proceeding is
  independently material."
- Distinguish risk-factor language ("could", "may") from disclosed actual events, and distinguish
  ongoing, historical, and completed matters.
- A 10-K for fiscal year N normally also contains comparative data for year N-1. Treat each
  requested period separately; only attribute a fact to a period when the evidence supports it.
  If a requested period is not supported, say it cannot be verified from the evidence.
- Absence of retrieved evidence is NOT evidence that something did not happen.

FISCAL PERIODS
- Document IDs look like COMPANY_YEAR[Qn]_TYPE. The year is only a label: read the evidence text to
  confirm which fiscal period it really covers. Never describe a figure as belonging to a period
  unless the evidence says so (a Q1 release labelled 2024 is not "Q1 of fiscal 2023").
- Companies name fiscal years differently (e.g. a retailer's year ending in January 2023 may be called
  "fiscal 2022"). If the evidence's fiscal-year name differs from the question's, say so in LIMITATIONS
  and state which period you used.
- Questions about "the end of" a fiscal year must be answered from the annual report or the Q4 /
  year-end earnings release, not from an interim quarter.

NUMBERS
- Financial tables state their scale in the header ("In thousands", "In millions"). Use the scale of the
  table you cite. If the question asks for another unit, convert, show the arithmetic, and round only
  at the end (e.g. $302,578 thousand = $302.578 million, about $303 million).
- When the question says to use a specific statement (balance sheet, income statement, cash flow
  statement), take the values from that statement's page. Pick the column for the requested fiscal year.
- Preserve original currencies and periods. Show inputs and formula for any calculation.
  Never combine figures from different periods without explaining the comparison.

OUTPUT FORMAT
If the evidence supports an answer:
STATUS: ANSWERED
ANSWER: <direct answer, then short explanation, per period when several are asked>
EVIDENCE:
- <finding> [DOCUMENT_ID, PDF page X]
LIMITATIONS: <gaps, period/label mismatches, or "None identified">

If it does not:
STATUS: REFUSED
ANSWER: The available filing evidence is insufficient to answer this question.
REASON: <what is missing>"""


def generate(question, evidence):
    if not evidence:
        return "STATUS: REFUSED\nANSWER: No relevant filing evidence was retrieved.\nREASON: Retrieval returned nothing."

    user_msg = (
        f"QUESTION:\n{question}\n\n"
        f"RETRIEVED SEC FILING EVIDENCE:\n{build_context(evidence)}\n\n"
        "Answer the exact question asked, period by period, using only this evidence."
    )
    resp = co.chat(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.0,
    )
    return resp.message.content[0].text


# --------------------------------------------------------------------------
# Exact page extraction from the answer's citations
# --------------------------------------------------------------------------
CITE_RE = re.compile(
    r"([A-Za-z0-9]+(?:_[A-Za-z0-9]+)+)\s*,\s*PDF\s*pages?\s*"
    r"(\d+(?:\s*(?:,|and|&|-|\u2013)\s*\d+)*)",
    re.I,
)


def parse_citations(answer):
    """
    Exact (document -> PDF pages) pairs from citations such as
    [PEPSICO_2022_10K, PDF page 26], [3M_2022_10K, PDF pages 48, 50 and 52]
    or [A_2024Q1_EARNINGS, PDF page 2; B_2023_10K, PDF page 41] (several documents in one bracket).
    """
    pages = defaultdict(set)
    for doc_id, raw in CITE_RE.findall(answer):
        for lo, hi in re.findall(r"(\d+)\s*[-\u2013]\s*(\d+)", raw):          # ranges: 25-27
            pages[doc_id].update(range(int(lo), int(hi) + 1))
        raw = re.sub(r"\d+\s*[-\u2013]\s*\d+", " ", raw)
        pages[doc_id].update(int(n) for n in re.findall(r"\d+", raw))
    return {d: sorted(p) for d, p in pages.items()}


# --------------------------------------------------------------------------
# End-to-end
# --------------------------------------------------------------------------
def ask_filings(
    question,
    role,
    k_per_query=30,
    top_n=8,
    verbose=True,
):
    filters = extract_filters(question, role)
    queries = rewrite_query(question)
    candidates = retrieve(queries, role, filters, k_per_query=k_per_query)
    evidence = rerank(question, candidates, top_n=top_n)   # top reranked chunks go straight to the LLM
    answer = generate(question, evidence)

    if verbose:
        print("Filters:", filters)
        print("Queries:")
        for q in queries:
            print("  -", q.replace("\n", " ")[:120])
        print(f"\nCandidates: {len(candidates)} | Evidence chunks: {len(evidence)}")
        for e in evidence:
            print(f"  #{e['rerank_rank']:<3} p{e['page']:<4} {e['chunk_id']:<32} rerank={e['rerank_score']:.4f}")
        print("Pages cited in answer:", parse_citations(answer))

    return {
        "answer": answer,
        "queries": queries,
        "filters": filters,
        "citations": sorted({(e["doc_id"], e["page"]) for e in evidence}),   # retrieved evidence pages
        "cited_pages": parse_citations(answer),                              # pages the answer actually cites
        "evidence": evidence,
    }


if __name__ == "__main__":
    question = (
    "What drove the increase in Ulta Beauty's merchandise inventories balance at end of FY2023"
    )
    result = ask_filings(question, role="consumer")
    print("\n" + "=" * 70)
    print(result["answer"])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 674.98it/s]


Filters: {'companies': ['Ulta Beauty'], 'years': [2023], 'statements': [], 'interim': False}
Queries:
  - What drove the increase in Ulta Beauty's merchandise inventories balance at end of FY2023
  - Ulta Beauty consolidated balance sheets 2023 merchandise inventories
  - Item 7. Management's Discussion and Analysis inventory management 2023
  - Item 7. Management's Discussion and Analysis merchandise inventories 2023
  - Item 1A. Risk Factors inventory management 2023

Candidates: 104 | Evidence chunks: 8
  #1   p41   ULTABEAUTY_2023_10K_p41_c0207    rerank=0.5515
  #2   p40   ULTABEAUTY_2023_10K_p40_c0203    rerank=0.5251
  #3   p41   ULTABEAUTY_2023_10K_p41_c0208    rerank=0.4644
  #4   p3    ULTABEAUTY_2023Q4_EARNINGS_p3_c0009 rerank=0.4076
  #5   p61   ULTABEAUTY_2023_10K_p61_c0274    rerank=0.3260
  #6   p38   ULTABEAUTY_2023_10K_p38_c0196    rerank=0.2758
  #7   p42   ULTABEAUTY_2023_10K_p42_c0209    rerank=0.2564
  #8   p44   ULTABEAUTY_2023_10K_p44_c0221    rerank=0.2400
Pages

Filters: {'companies': ['Ulta Beauty'], 'years': [2023], 'statements': []}
Queries:
  - What drove the increase in Ulta Beauty's merchandise inventories balance at end of FY2023
  - Ulta Beauty consolidated balance sheets 2023 merchandise inventories
  - Item 7. Management's Discussion and Analysis inventory management 2023
  - Item 7. Management's Discussion and Analysis merchandise inventories 2023
  - Item 1A. Risk Factors inventory management 2023

Candidates: 122 | Evidence chunks: 8
  #1   p2    ULTABEAUTY_2024Q1_EARNINGS_p2_c0005 rerank=0.6846
  #2   p3    ULTABEAUTY_2024Q2_EARNINGS_p3_c0009 rerank=0.6789
  #3   p41   ULTABEAUTY_2023_10K_p41_c0207    rerank=0.5515
  #4   p40   ULTABEAUTY_2023_10K_p40_c0203    rerank=0.5251
  #5   p41   ULTABEAUTY_2023_10K_p41_c0208    rerank=0.4644
  #6   p3    ULTABEAUTY_2023Q4_EARNINGS_p3_c0009 rerank=0.4076
  #7   p61   ULTABEAUTY_2023_10K_p61_c0274    rerank=0.3260
  #8   p38   ULTABEAUTY_2023_10K_p38_c0196    rerank=0.2758
Pages cited in an